# PAC2026: RGB 상자 검출 + SAM 분리 — Colab GPU
1단계는 RGB 사진에서 상자의 bbox/마스크를 검증합니다. 로봇/직렬 포트/로컬 서버 연결은 없습니다.

런타임 → 런타임 유형 변경 → **T4 GPU** 선택 후 위에서부터 실행하세요. GPU가 없으면 중단합니다. 자신의 카메라에서 RGB 사진만 업로드하세요. 개인 Drive 전체를 마운트하거나 HF 토큰을 요구하지 않습니다.

Colab의 127.0.0.1은 현장 노트북이 아닙니다. 실제 카메라 스트림·깊이·로봇 제어 연결은 다음 단계이며 클라우드 추론 시간을 실제 제어 지연으로 취급하지 않습니다. 기존 LeRobot 정책 학습 노트북과 별도입니다.

In [ ]:
import os, sys, json, time, subprocess
from pathlib import Path
import torch
assert torch.cuda.is_available(), "GPU 없음: 런타임 유형을 T4 GPU로 변경하세요"
print("GPU:", torch.cuda.get_device_name(0), "torch:", torch.__version__)
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], check=True)
Path("/content/pac_rgb").mkdir(exist_ok=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==5.13.0", "huggingface_hub>=1.3,<2", "httpx>=0.28,<1", "Pillow>=10", "opencv-python-headless>=4.10,<5"], check=True)

## 공유 구현 사용
아래 파일은 `tiltis/codex/rgb_box`의 구현을 그대로 포함합니다. 수정 시 `make_colab.py`를 다시 실행해 노트북을 갱신하세요.

In [ ]:
%%writefile /content/pac_rgb/device.py
"""Explicit compute selection; requesting CUDA never silently uses the CPU."""


def select_device(requested, torch):
    if requested not in ("cpu", "cuda"):
        raise ValueError("device must be cpu or cuda")
    if requested == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA GPU unavailable: select a GPU runtime in Colab first")
    return requested


def synchronize(torch, device):
    if device == "cuda":
        torch.cuda.synchronize()


In [ ]:
%%writefile /content/pac_rgb/detector.py
"""Stage 1: RGB box candidates only. No depth, calibration or motor imports."""
from __future__ import annotations

import math
import time

from device import select_device, synchronize

MODEL_ID = "IDEA-Research/grounding-dino-tiny"
MODEL_REVISION = "a2bb814dd30d776dcf7e30523b00659f4f141c71"
DEFAULT_LABELS = ["a cardboard box", "a white cardboard box"]


def iou(a, b):
    overlap = max(0, min(a[2], b[2]) - max(a[0], b[0])) * max(0, min(a[3], b[3]) - max(a[1], b[1]))
    aa, bb = (a[2] - a[0]) * (a[3] - a[1]), (b[2] - b[0]) * (b[3] - b[1])
    return overlap / (aa + bb - overlap)


def candidates(boxes, scores, labels, size, threshold=0.4, nms_iou=0.5):
    """Validate model coordinates; merge duplicate text matches, retain separate objects."""
    if len(boxes) != len(scores) or len(boxes) != len(labels):
        raise ValueError("model result lengths differ")
    width, height = size
    result = []
    for box, score, label in zip(boxes, scores, labels):
        coords, score = [float(v) for v in box], float(score)
        if len(coords) != 4 or not all(math.isfinite(v) for v in [*coords, score]):
            raise ValueError("nonfinite model result")
        if not 0 <= score <= 1:
            raise ValueError("invalid model score")
        if score < threshold:
            continue
        x0, y0, x1, y1 = coords
        if not (0 <= x0 < x1 <= width and 0 <= y0 < y1 <= height):
            raise ValueError("model box outside source image")
        result.append({"label": str(label), "score": round(score, 4),
                       "bbox_px": [round(v, 2) for v in coords],
                       "center_px": [round((x0 + x1) / 2, 2), round((y0 + y1) / 2, 2)]})
    kept = []
    for item in sorted(result, key=lambda v: v["score"], reverse=True):
        if not any(iou(item["bbox_px"], v["bbox_px"]) >= nms_iou for v in kept):
            kept.append(item)
    return kept


class BoxDetector:
    def __init__(self, model_dir=None, threads=4, shortest_edge=512, labels=None, threshold=0.4, device="cpu"):
        import torch
        from transformers import AutoModelForZeroShotObjectDetection, AutoProcessor
        torch.set_num_threads(threads)
        self.device = select_device(device, torch)
        self.torch, self.labels, self.threshold = torch, labels or DEFAULT_LABELS, threshold
        model = model_dir or MODEL_ID
        options = {"local_files_only": True}
        if model_dir is None:
            options["revision"] = MODEL_REVISION
        self.processor = AutoProcessor.from_pretrained(model, **options)
        self.model = AutoModelForZeroShotObjectDetection.from_pretrained(
            model, **options, use_safetensors=True, disable_custom_kernels=True).to(self.device).eval()
        self.size = {"shortest_edge": shortest_edge, "longest_edge": max(shortest_edge, 768)}

    def detect(self, image):
        synchronize(self.torch, self.device)
        t0 = time.perf_counter()
        inputs = self.processor(images=image, text=[self.labels], size=self.size, return_tensors="pt").to(self.device)
        with self.torch.inference_mode():
            output = self.model(**inputs)
        parsed = self.processor.post_process_grounded_object_detection(
            output, inputs.input_ids, threshold=self.threshold, text_threshold=0.25,
            target_sizes=[image.size[::-1]])[0]
        # Text labels moved to text_labels in newer Transformers versions.
        labels = parsed.get("text_labels", parsed.get("labels"))
        boxes = candidates(parsed["boxes"].tolist(), parsed["scores"].tolist(), labels,
                           image.size, self.threshold)
        synchronize(self.torch, self.device)
        return {"stage": "rgb_box_detection", "frame": "rgb_preview_px", "motion_enabled": False,
                "robot_ready": False, "depth_association": "not_established", "model": MODEL_ID,
                "model_revision": MODEL_REVISION, "device": self.device, "image_size_px": list(image.size),
                "found": bool(boxes), "candidate_count": len(boxes), "detections": boxes,
                "status": "single_box_candidate" if len(boxes) == 1 else ("multiple_candidates" if boxes else "no_box"),
                "inference_ms": round((time.perf_counter() - t0) * 1000, 1)}


In [ ]:
%%writefile /content/pac_rgb/segment.py
"""Optional compact SAM masks, prompted automatically by RGB detector boxes."""
from __future__ import annotations

import time

import numpy as np

from device import select_device, synchronize

MODEL_ID = "Zigeng/SlimSAM-uniform-77"
MODEL_REVISION = "79c09c1ce6b4ae51f00634ed171d9b8e888f6911"


def mask_geometry(mask, size):
    a = np.asarray(mask)
    if a.shape != (size[1], size[0]) or a.dtype != np.bool_:
        raise ValueError("mask must be boolean and match the source RGB dimensions")
    y, x = np.nonzero(a)
    if not len(x):
        raise ValueError("SAM returned an empty mask")
    return {"mask_area_px": int(len(x)), "mask_center_px": [round(float(x.mean()), 2), round(float(y.mean()), 2)],
            "mask_bbox_px": [int(x.min()), int(y.min()), int(x.max()) + 1, int(y.max()) + 1]}


class BoxSegmenter:
    def __init__(self, device="cpu"):
        import torch
        from transformers import SamModel, SamProcessor
        self.torch = torch
        self.device = select_device(device, torch)
        options = {"revision": MODEL_REVISION, "local_files_only": True}
        self.processor = SamProcessor.from_pretrained(MODEL_ID, **options)
        self.model = SamModel.from_pretrained(MODEL_ID, **options, use_safetensors=True).to(self.device).eval()

    def segment(self, image, detections):
        if not detections:
            return [], {"model": MODEL_ID, "revision": MODEL_REVISION, "device": self.device, "inference_ms": 0}
        synchronize(self.torch, self.device)
        t0 = time.perf_counter()
        inputs = self.processor(images=image, input_boxes=[[v["bbox_px"] for v in detections]], return_tensors="pt").to(self.device)
        with self.torch.inference_mode():
            out = self.model(**inputs)
        masks = self.processor.post_process_masks(out.pred_masks.cpu(), inputs["original_sizes"].cpu(),
                                                  inputs["reshaped_input_sizes"].cpu())[0]
        selected = []
        for i, detection in enumerate(detections):
            best = int(out.iou_scores[0, i].argmax())
            mask = masks[i, best].numpy().astype(bool)
            detection.update(mask_geometry(mask, image.size), sam_iou_score=round(float(out.iou_scores[0, i, best]), 4))
            selected.append(mask)
        synchronize(self.torch, self.device)
        return selected, {"model": MODEL_ID, "revision": MODEL_REVISION,
                          "device": self.device, "inference_ms": round((time.perf_counter() - t0) * 1000, 1)}


In [ ]:
%%writefile /content/pac_rgb/source.py
"""Read the existing sensor preview; never open another camera handle."""
from __future__ import annotations

import io
import time

import httpx
import numpy as np
from PIL import Image, ImageDraw


def rgb_panel(data, rgb_width=480):
    image = Image.open(io.BytesIO(data)).convert("RGB")
    if type(rgb_width) is not int or rgb_width <= 0 or rgb_width >= image.width:
        raise ValueError("explicit RGB panel width required for composite preview")
    return image.crop((0, 0, rgb_width, image.height))


def fetch_rgb(client, url, rgb_width=480):
    response = client.get(url)
    response.raise_for_status()
    image = rgb_panel(response.content, rgb_width)
    return image, {"received_at_s": time.time(), "capture_time_verified": False,
                   "source": "existing_sensor_composite_preview", "rgb_panel_width_px": rgb_width,
                   "original_rgb_size_px": None}


def annotate(image, result, masks=()):
    out = image.copy()
    if masks:
        rgb = np.asarray(out).copy()
        for mask in masks:
            rgb[mask] = (rgb[mask] * 0.65 + np.array([255, 210, 0]) * 0.35).astype(np.uint8)
        out = Image.fromarray(rgb)
    draw = ImageDraw.Draw(out)
    for i, item in enumerate(result["detections"], 1):
        box, center = item["bbox_px"], item["center_px"]
        draw.rectangle(box, outline="#00ff77", width=3)
        x, y = center
        draw.line((x - 7, y, x + 7, y), fill="#ff66ff", width=2)
        draw.line((x, y - 7, x, y + 7), fill="#ff66ff", width=2)
        label = f"BOX {i} | score {item['score']:.2f}" + (" | SAM" if masks else "")
        left, top = max(0, box[0]), max(36, box[1] - 16)
        draw.rectangle((left, top, min(out.width, left + 185), top + 15), fill="#003b23")
        draw.text((left + 3, top + 1), label, fill="white")
    footer = f"RGB ONLY | {result['candidate_count']} candidates | NO ROBOT COMMANDS"
    draw.rectangle((0, out.height - 20, out.width, out.height), fill="#102030")
    draw.text((6, out.height - 16), footer, fill="white")
    return out


In [ ]:
%%writefile /content/pac_rgb/top_face.py
"""SAM face proposals for previews. RGB alone does not confirm which face is the top."""
from __future__ import annotations

import time
import cv2
import numpy as np
from PIL import Image, ImageDraw

from device import synchronize
from segment import mask_geometry


def face_prompts(box, size):
    x0, y0, x1, y1 = np.asarray(box, float)
    if not np.isfinite([x0, y0, x1, y1]).all() or not (0 <= x0 < x1 <= size[0] and 0 <= y0 < y1 <= size[1]):
        raise ValueError("face prompt box outside source RGB")
    w, h, cx = x1 - x0, y1 - y0, (x0 + x1) / 2
    return [[[cx, y0 + r*h], [cx, y0 + .82*h], [x0 + .15*w, y0 + .65*h], [x0 + .85*w, y0 + .65*h]]
            for r in (.06, .12, .2, .3)]


def face_geometry(mask, whole_mask, points):
    mask_geometry(mask, (whole_mask.shape[1], whole_mask.shape[0]))
    mask_geometry(whole_mask, (whole_mask.shape[1], whole_mask.shape[0]))
    pts = np.asarray(points, float)
    if pts.shape != (4, 2) or not np.isfinite(pts).all():
        raise ValueError("one positive and three negative RGB prompts required")
    xy = np.rint(pts).astype(int)
    if (xy < 0).any() or (xy[:, 0] >= mask.shape[1]).any() or (xy[:, 1] >= mask.shape[0]).any():
        raise ValueError("prompt outside source RGB")
    values = mask[xy[:, 1], xy[:, 0]].tolist()
    area = int(mask.sum())
    fraction = area / int(whole_mask.sum())
    overlap = float((mask & whole_mask).sum()) / area
    if values != [True, False, False, False] or not .05 < fraction < .65 or overlap < .95:
        return None
    contours, _ = cv2.findContours(mask.astype("uint8"), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    hull = cv2.convexHull(max(contours, key=cv2.contourArea))
    quad = cv2.approxPolyDP(hull, .03 * cv2.arcLength(hull, True), True).reshape(-1, 2)
    if len(quad) != 4:
        return None
    filled = np.zeros(mask.shape, "uint8")
    cv2.fillConvexPoly(filled, quad, 1)
    q_iou = float((mask & (filled > 0)).sum()) / float((mask | (filled > 0)).sum())
    if q_iou < .7:
        return None
    edges = np.roll(quad, -1, axis=0).astype(float) - quad
    if np.linalg.norm(edges, axis=1).min() < 5:
        return None
    # Perspective gives two projected edge directions, not a physical robot yaw.
    directions = [edges[i] / np.linalg.norm(edges[i]) - edges[i + 2] / np.linalg.norm(edges[i + 2]) for i in (0, 1)]
    angles = [round(float(np.degrees(np.arctan2(v[1], v[0])) % 180), 2) for v in directions]
    return {"quad_px": quad.tolist(), "edge_angles_image_deg": angles,
            "quad_iou": round(q_iou, 4), "whole_mask_overlap": round(overlap, 4),
            "whole_mask_fraction": round(fraction, 4), "top_confirmed": False,
            "frame": "rgb_px_y_down", "robot_yaw_deg": None}


def propose_faces(segmenter, image, detection, whole_mask):
    prompts = face_prompts(detection["bbox_px"], image.size)
    synchronize(segmenter.torch, segmenter.device)
    t0 = time.perf_counter()
    inputs = segmenter.processor(images=image, input_points=[prompts],
                                 input_labels=[[[1, 0, 0, 0]] * len(prompts)], return_tensors="pt").to(segmenter.device)
    with segmenter.torch.inference_mode():
        out = segmenter.model(**inputs)
    masks = segmenter.processor.post_process_masks(out.pred_masks.cpu(), inputs["original_sizes"].cpu(),
                                                  inputs["reshaped_input_sizes"].cpu())[0]
    proposed, selected_masks = [], []
    for p, points in enumerate(prompts):
        for m in range(masks.shape[1]):
            mask = masks[p, m].numpy().astype(bool)
            if not mask.any():
                continue
            geometry = face_geometry(mask, whole_mask, points)
            if geometry is None:
                continue
            if any((mask & old).sum() / (mask | old).sum() >= .85 for old in selected_masks):
                continue
            proposed.append(dict(geometry, sam_iou_score=round(float(out.iou_scores[0, p, m]), 4), prompt=p, mask=m))
            selected_masks.append(mask)
    synchronize(segmenter.torch, segmenter.device)
    return proposed, selected_masks, {"inference_ms": round((time.perf_counter() - t0) * 1000, 1),
                                      "device": segmenter.device, "selected_top": None,
                                      "selection_requires": "registered_depth_top_plane", "motion_enabled": False}


def annotate_face(image, candidate, mask):
    rgb = np.asarray(image).copy()
    rgb[mask] = (rgb[mask] * .5 + np.array([0, 180, 255]) * .5).astype("uint8")
    vis = Image.fromarray(rgb)
    draw = ImageDraw.Draw(vis)
    quad = candidate["quad_px"]
    draw.line([tuple(v) for v in quad] + [tuple(quad[0])], fill="red", width=3)
    text = f"FACE CANDIDATE | image angles {candidate['edge_angles_image_deg']} | depth required"
    draw.rectangle((0, image.height - 20, image.width, image.height), fill="#102030")
    draw.text((6, image.height - 16), text, fill="white")
    return vis


## 공개 모델 다운로드 및 GPU 로드
모델 revision 고정, safetensors만 사용. 모델 가중치 로드 시간은 사진별 추론 시간에서 제외합니다.

In [ ]:
sys.path.insert(0, "/content/pac_rgb")
from detector import BoxDetector, MODEL_ID as DINO_ID, MODEL_REVISION as DINO_REV
from segment import BoxSegmenter, MODEL_ID as SAM_ID, MODEL_REVISION as SAM_REV
from source import annotate
from PIL import Image
from IPython.display import display
import shutil, transformers
hf = shutil.which("hf")
assert hf, "hf CLI installation missing"
subprocess.run([hf, "download", DINO_ID, "model.safetensors", "config.json", "preprocessor_config.json", "tokenizer.json", "tokenizer_config.json", "special_tokens_map.json", "added_tokens.json", "vocab.txt", "--revision", DINO_REV], check=True)
subprocess.run([hf, "download", SAM_ID, "model.safetensors", "config.json", "preprocessor_config.json", "--revision", SAM_REV], check=True)
detector = BoxDetector(device="cuda")
segmenter = BoxSegmenter(device="cuda")
assert next(detector.model.parameters()).is_cuda and next(segmenter.model.parameters()).is_cuda
print("GPU models loaded:", detector.device, segmenter.device)

## RGB 사진 업로드
검증할 실제 RGB 사진을 선택하세요. 3카메라 합성 사진 대신 RGB 패널만 선택하세요. 파일 선택 위젯이 동작하지 않으면 왼쪽 **파일 → 세션 저장소에 업로드**로 사진을 올리고 아래 `IMAGE_FILES`에 파일 이름을 넣으세요. 업로드한 파일은 이 Colab 세션의 추론 입력이며 GitHub에는 넣지 않습니다.

In [ ]:
from google.colab import files
IMAGE_FILES = []  # 파일 패널로 올렸다면 ["box1.png", "box2.png"]처럼 지정
if IMAGE_FILES:
    assert all(Path(name).is_file() for name in IMAGE_FILES), "파일 패널에 먼저 업로드하세요"
    uploaded = {name: b"" for name in IMAGE_FILES}
else:
    uploaded = files.upload()
assert uploaded, "검증할 RGB 사진이 필요합니다"
print("RGB files:", list(uploaded))

## 검출·분리·저장
사진마다 RGB 픽셀 bbox/중심/마스크와 추론 시간을 표시합니다. GPU 워밍업 한 장은 시간 비교에서 제외하고 모든 입력을 다시 처리합니다. CUDA 동기화 후 시간을 기록합니다. 모델 점수/추정 IoU는 측정한 정확도가 아닙니다.

In [ ]:
output_dir = Path("/content/pac_rgb_results")
output_dir.mkdir(exist_ok=True)
names = [name for name in uploaded if Path(name).suffix.lower() in {".png", ".jpg", ".jpeg", ".webp"}]
assert names, "지원되는 RGB 이미지가 없습니다"
warm = Image.open(names[0]).convert("RGB")
warm_result = detector.detect(warm)
segmenter.segment(warm, warm_result["detections"])
summary = []
for index, name in enumerate(names, 1):
    image = Image.open(name).convert("RGB")
    result = detector.detect(image)
    result["frame"] = "uploaded_rgb_px"
    result["source_metadata"] = {"source": "colab_uploaded_rgb", "filename": Path(name).name, "capture_time_verified": False}
    masks, result["segmentation"] = segmenter.segment(image, result["detections"])
    stem = f"{index:03}_{Path(name).stem}"
    for number, mask in enumerate(masks, 1):
        mask_name = f"{stem}_mask{number}.png"
        Image.fromarray(mask.astype("uint8") * 255).save(output_dir / mask_name)
        result["detections"][number - 1]["mask_file"] = mask_name
    annotated = annotate(image, result, masks)
    annotated.save(output_dir / f"{stem}.png")
    (output_dir / f"{stem}.json").write_text(json.dumps(result, ensure_ascii=False, indent=2, allow_nan=False))
    row = {"image": Path(name).name, "candidates": result["candidate_count"], "dino_ms": result["inference_ms"], "sam_ms": result["segmentation"]["inference_ms"], "detections": result["detections"]}
    summary.append(row)
    print(json.dumps(row, ensure_ascii=False))
    display(annotated.resize((min(image.width, 800), round(image.height * min(image.width, 800) / image.width))))
manifest = {"gpu": torch.cuda.get_device_name(0), "torch": torch.__version__, "transformers": transformers.__version__, "motion_enabled": False, "robot_ready": False, "warmup_excluded": True, "results": summary}
(output_dir / "summary.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2, allow_nan=False))
print("GPU VALIDATION COMPLETE", manifest["gpu"], "images:", len(summary))

## SAM 면 후보와 영상 각도
상자 전체 마스크와 윗면은 다릅니다. 자동 point/negative prompts로 면 후보·사각형·두 영상 변의 각도를 표시합니다. 테이프나 앞면 후보도 나올 수 있어 RGB 점수로 윗면을 확정하지 않습니다. `sam_depth.py`에서 검증된 RGB–depth 정합·동시 촬영·책상 평면과 상자 실측 크기를 확인한 후 로봇 방향에 사용할 3D 축을 생성합니다. 출력의 영상 각도는 로봇 yaw가 아닙니다.

In [ ]:
"""Colab cell body: loaded detector/segmenter, uploaded RGB files, CUDA runtime."""
from top_face import propose_faces, annotate_face
from google.colab import files
import json, shutil
from pathlib import Path
from PIL import Image
from IPython.display import display

face_dir = Path('/content/pac_sam_faces')
face_dir.mkdir(exist_ok=True)
face_summary = []
for index, name in enumerate(uploaded, 1):
    image = Image.open(name).convert('RGB')
    detection = detector.detect(image)
    if detection['candidate_count'] != 1:
        face_summary.append({'image': name, 'reason': 'single_box_not_confirmed', 'faces': []})
        continue
    whole, _ = segmenter.segment(image, detection['detections'])
    faces, masks, info = propose_faces(segmenter, image, detection['detections'][0], whole[0])
    for number, (candidate, mask) in enumerate(zip(faces, masks), 1):
        stem = f'{index:02}_{Path(name).stem}_face{number}'
        Image.fromarray(mask.astype('uint8') * 255).save(face_dir / f'{stem}_mask.png')
        vis = annotate_face(image, candidate, mask)
        vis.save(face_dir / f'{stem}.png')
        candidate['mask_file'] = f'{stem}_mask.png'
        candidate['preview_file'] = f'{stem}.png'
        display(vis.resize((min(image.width, 800), round(image.height * min(image.width, 800) / image.width))))
    row = dict(image=name, faces=faces, **info)
    face_summary.append(row)
    print(json.dumps(row, ensure_ascii=False))
manifest = {'gpu': torch.cuda.get_device_name(0), 'motion_enabled': False, 'robot_ready': False,
            'top_confirmed': False, 'results': face_summary,
            'note': 'RGB candidates include tape/front faces. Registered depth must select the top plane.'}
(face_dir / 'summary.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2, allow_nan=False))
print('SAM FACE GPU COMPLETE', manifest['gpu'], 'images:', len(face_summary), '| DEPTH REQUIRED')
files.download(shutil.make_archive('/content/pac_sam_faces_gpu', 'zip', face_dir))


## 상자 전체 결과 다운로드
PNG/개별 마스크/JSON/환경·시간 요약을 보관합니다. 노트북 실행 결과에 RGB 사진이 표시되므로 공유하기 전에 출력 내용을 확인하세요. 검증 종료 후 런타임 연결 해제로 GPU 사용을 마칩니다.

In [ ]:
archive = shutil.make_archive("/content/pac_rgb_gpu_results", "zip", output_dir)
files.download(archive)